**ENCRYPTION CODE**

In [ ]:
!pip -q install blake3 opencv-python pillow scikit-image

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))

original_image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)

if original_image is None:
    raise ValueError("Unable to read the uploaded image.")


if original_image.ndim == 2:
    image = original_image
    channels = 1

elif original_image.ndim == 3:
    image = original_image
    channels = original_image.shape[2]

else:
    raise ValueError("Unsupported image dimension.")


if image.dtype != np.uint8:
    image = np.clip(image, 0, 255).astype(np.uint8)


H, W = image.shape[:2]
C = channels
N = H * W * C


plt.figure(figsize=(7, 7))

if C == 1:
    plt.imshow(image, cmap="gray")
else:
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

plt.title("Original Image")
plt.axis("off")
plt.show()


print("Image loaded successfully.")
print(f"Shape    : {image.shape}")
print(f"Height   : {H}")
print(f"Width    : {W}")
print(f"Channels : {C}")
print(f"Data type: {image.dtype}")
print(f"N        : {N}")
print(f"Range    : [{image.min()}, {image.max()}]")


X = image.reshape(-1).copy()

print(f"Flattened X shape : {X.shape}")
print(f"X ∈ Z_256^N      : {np.all((X >= 0) & (X <= 255))}")


import hashlib

def extract_environmental_noise(length=64):
    """Collect plaintext-independent runtime/environmental noise."""
    noise = bytearray()

    while len(noise) < length:
        noise.extend(os.urandom(64))

    return bytes(noise[:length])


def generate_session_key():
    """Generate a one-time 256-bit session key from conditioned noise."""
    R = extract_environmental_noise(64)


    Z = hashlib.sha512(R).digest()


    Ks = hashlib.sha256(
        Z + b"SESSION_KEY"
    ).digest()

    return R, Z, Ks


R, Z, Ks = generate_session_key()

print("Environmental noise length :", len(R), "bytes")
print("Conditioned noise length   :", len(Z), "bytes")
print("Session key length         :", len(Ks) * 8, "bits")
print("Session key                :", Ks.hex())



def domain_kdf(Ks, domain):
    return hashlib.sha256(
        Ks + domain.encode("utf-8")
    ).digest()


KCNN   = domain_kdf(Ks, "CNN")
KBNM   = domain_kdf(Ks, "BNM")
KHENON = domain_kdf(Ks, "HENON")
KP     = domain_kdf(Ks, "PERM")
KF     = domain_kdf(Ks, "FDIFF")
KB     = domain_kdf(Ks, "BDIFF")


print("KCNN   :", KCNN.hex())
print("KBNM   :", KBNM.hex())
print("KHENON :", KHENON.hex())
print("KP     :", KP.hex())
print("KF     :", KF.hex())
print("KB     :", KB.hex())


import torch
import torch.nn as nn

def generate_cnn_seed(KCNN):
    X_CNN = np.frombuffer(
        hashlib.sha256(KCNN).digest(),
        dtype=np.uint8
    ).astype(np.float32)

    X_CNN = torch.from_numpy(X_CNN).reshape(1, 1, 32)

    class LightweightCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv1d(1, 8, 3, padding=1)
            self.conv2 = nn.Conv1d(8, 16, 3, padding=1)
            self.fc = nn.Linear(16, 32)

        def forward(self, x):
            x = torch.relu(self.conv1(x))
            x = torch.relu(self.conv2(x))
            x = x.mean(dim=2)
            return self.fc(x)

    torch.manual_seed(int.from_bytes(KCNN[:8], "big"))
    cnn = LightweightCNN().eval()

    with torch.no_grad():
        z = cnn(X_CNN)

    seed_bytes = hashlib.sha256(
        z.numpy().tobytes() + KCNN
    ).digest()

    return seed_bytes


S_CNN = generate_cnn_seed(KCNN)

print("CNN input shape :", (32, 1))
print("S_CNN size      :", len(S_CNN) * 8, "bits")
print("S_CNN            :", S_CNN.hex())



def generate_dynamic_parameters(S_CNN, N):
    a = np.empty(N, dtype=np.uint8)
    b = np.empty(N, dtype=np.uint8)

    for i in range(N):
        G_i = hashlib.sha256(
            S_CNN + i.to_bytes(8, "big")
        ).digest()

        q_i = int.from_bytes(G_i[:8], "big")
        b_i = int.from_bytes(G_i[8:16], "big")

        a[i] = 2 * (q_i % 128) + 1
        b[i] = b_i % 256

    return a, b


a, b = generate_dynamic_parameters(S_CNN, N)

print("Dynamic parameters generated.")
print("a shape :", a.shape)
print("b shape :", b.shape)
print("a range :", int(a.min()), "to", int(a.max()))
print("b range :", int(b.min()), "to", int(b.max()))
print("All a_i odd :", bool(np.all(a % 2 == 1)))
print("All gcd(a_i, 256) = 1 :", bool(np.all(np.gcd(a.astype(np.int16), 256) == 1)))


def modular_kernel_transform(X, a, b):
    return ((a.astype(np.uint16) * X.astype(np.uint16) +
             b.astype(np.uint16)) & 255).astype(np.uint8)


T = modular_kernel_transform(X, a, b)

print("Reversible modular transformation completed.")
print("Input  X shape :", X.shape)
print("Output T shape :", T.shape)
print("T range        :", int(T.min()), "to", int(T.max()))
print("T dtype        :", T.dtype)
print("Transformation : ti = (ai * xi + bi) mod 256")



def generate_bnm_sequence(KBNM, N):
    seed = hashlib.sha256(KBNM + b"BNM_STATE").digest()

    x = int.from_bytes(seed[:8], "big") / 2**64
    y = int.from_bytes(seed[8:16], "big") / 2**64
    z = int.from_bytes(seed[16:24], "big") / 2**64

    x = 0.1 + 0.8 * x
    y = 0.1 + 0.8 * y
    z = 0.1 + 0.8 * z

    sequence = np.empty(N, dtype=np.float64)

    for i in range(N):
        x_new = np.sin(3.99 * y) + 1.99 * np.cos(z)
        y_new = np.sin(3.99 * z) + 1.99 * np.cos(x)
        z_new = np.sin(3.99 * x) + 1.99 * np.cos(y)

        x = x_new - np.floor(x_new)
        y = y_new - np.floor(y_new)
        z = z_new - np.floor(z_new)

        sequence[i] = (x + y + z) / 3.0

    return sequence


B = generate_bnm_sequence(KBNM, N)

KB_stream = np.floor(B * 256).astype(np.uint8)

print("BNM sequence generated.")
print("B shape       :", B.shape)
print("KB shape      :", KB_stream.shape)
print("KB dtype      :", KB_stream.dtype)
print("KB range      :", int(KB_stream.min()), "to", int(KB_stream.max()))




def generate_henon_sequence(KHENON, N):
    seed = hashlib.sha512(KHENON + b"HENON_STATE").digest()

    x = int.from_bytes(seed[0:8], "big") / 2**64
    y = int.from_bytes(seed[8:16], "big") / 2**64

    x = 0.1 + 0.8 * x
    y = 0.1 + 0.8 * y

    a_h = 1.4
    b_h = 0.3

    sequence = np.empty(N, dtype=np.float64)

    for i in range(N):
        x_new = 1.0 - a_h * x * x + y
        y_new = b_h * x

        x = x_new
        y = y_new

        sequence[i] = x

    return sequence


H = generate_henon_sequence(KHENON, N)

KH_stream = (
    np.floor(
        (np.abs(H) % 1.0) * 256.0
    ).astype(np.uint8)
)

print("Henon sequence generated.")
print("H shape       :", H.shape)
print("KH shape      :", KH_stream.shape)
print("KH dtype      :", KH_stream.dtype)
print("KH range      :", int(KH_stream.min()), "to", int(KH_stream.max()))




def generate_hybrid_keystream(KB_stream, KH_stream, KF, KB, N):
    # Ki = KB(i) XOR KH(i)
    K = np.bitwise_xor(KB_stream, KH_stream)

    kF = np.empty(N, dtype=np.uint8)
    kB = np.empty(N, dtype=np.uint8)

    for i in range(N):
        index = i.to_bytes(8, "big")

        # kiF = PRF(Ki, "F" || i) mod 256
        kF[i] = hashlib.sha256(
            K[i:i+1].tobytes() + KF + b"F" + index
        ).digest()[0]

        # kiB = PRF(Ki, "B" || i) mod 256
        kB[i] = hashlib.sha256(
            K[i:i+1].tobytes() + KB + b"B" + index
        ).digest()[0]

    return K, kF, kB


K, kF, kB = generate_hybrid_keystream(
    KB_stream,
    KH_stream,
    KF,
    KB,
    N
)

print("Hybrid chaotic keystream generated.")
print("K  shape :", K.shape)
print("kF shape :", kF.shape)
print("kB shape :", kB.shape)
print("K  dtype :", K.dtype)
print("kF dtype :", kF.dtype)
print("kB dtype :", kB.dtype)




def regenerate_diffusion_streams(K, N):
    kF = np.empty(N, dtype=np.uint8)
    kB = np.empty(N, dtype=np.uint8)

    for i in range(N):
        idx = i.to_bytes(8, "big")
        key_i = K[i:i + 1].tobytes()

        kF[i] = hashlib.sha256(
            key_i + b"F" + idx
        ).digest()[0]

        kB[i] = hashlib.sha256(
            key_i + b"B" + idx
        ).digest()[0]

    return kF, kB


kF, kB = regenerate_diffusion_streams(K, N)

print("Forward stream regenerated :", kF.shape)
print("Backward stream regenerated:", kB.shape)
print("kF dtype                  :", kF.dtype)
print("kB dtype                  :", kB.dtype)



def fisher_yates_permutation(N, KP):
    P = np.arange(N, dtype=np.int64)

    for i in range(N - 1, 0, -1):
        r = hashlib.sha256(
            KP + i.to_bytes(8, "big")
        ).digest()

        j = int.from_bytes(r[:8], "big") % (i + 1)

        P[i], P[j] = P[j], P[i]

    return P


def apply_global_permutation(T, P):
    return T[P]


P = fisher_yates_permutation(N, KP)
Y = apply_global_permutation(T, P)

print("Fisher-Yates global permutation completed.")
print("Permutation shape :", P.shape)
print("Permutation valid :", bool(np.array_equal(np.sort(P), np.arange(N))))
print("Y shape           :", Y.shape)
print("Y dtype           :", Y.dtype)




def block_channel_permutation(Y, H, W, channels, KP):

    H = int(H)
    W = int(W)
    channels = int(channels)

    image = Y.reshape(H, W, channels) if channels > 1 else Y.reshape(H, W)


    if channels > 1:
        channel_key = hashlib.sha256(
            KP + b"CHANNEL"
        ).digest()

        channel_order = np.arange(channels, dtype=np.int64)

        for i in range(channels - 1, 0, -1):
            j = int.from_bytes(
                channel_key[i % 32:(i % 32) + 1],
                "big"
            ) % (i + 1)

            channel_order[i], channel_order[j] = (
                channel_order[j],
                channel_order[i]
            )

        image = image[:, :, channel_order]


    block_size = 16

    if H >= block_size and W >= block_size:

        h_blocks = H // block_size
        w_blocks = W // block_size
        total_blocks = h_blocks * w_blocks

        order = np.arange(total_blocks, dtype=np.int64)

        block_key = hashlib.sha256(
            KP + b"BLOCK"
        ).digest()

        for i in range(total_blocks - 1, 0, -1):
            counter = i.to_bytes(8, "big")

            j = int.from_bytes(
                hashlib.sha256(
                    block_key + counter
                ).digest()[:8],
                "big"
            ) % (i + 1)

            order[i], order[j] = order[j], order[i]

        result = image.copy()

        for dst, src in enumerate(order):

            dst_r = (dst // w_blocks) * block_size
            dst_c = (dst % w_blocks) * block_size

            src_r = (src // w_blocks) * block_size
            src_c = (src % w_blocks) * block_size

            result[
                dst_r:dst_r + block_size,
                dst_c:dst_c + block_size
            ] = image[
                src_r:src_r + block_size,
                src_c:src_c + block_size
            ]

        image = result

    Z = image.reshape(-1).astype(np.uint8)

    return Z



H_img, W_img = image.shape[:2]
C_img = 1 if image.ndim == 2 else image.shape[2]

Z = block_channel_permutation(
    Y,
    H_img,
    W_img,
    C_img,
    KP
)

print("Block / channel permutation completed.")
print("Z shape :", Z.shape)
print("Z dtype :", Z.dtype)
print("Z range :", int(Z.min()), "to", int(Z.max()))



def forward_diffusion(Z, kF):
    F = np.empty_like(Z, dtype=np.uint8)

    F[0] = (int(Z[0]) + int(kF[0])) & 255

    for i in range(1, len(Z)):
        F[i] = (
            int(Z[i])
            + int(kF[i])
            + int(F[i - 1])
        ) & 255

    return F


F = forward_diffusion(Z, kF)

print("Forward diffusion completed correctly.")
print("F shape :", F.shape)
print("F dtype :", F.dtype)



def backward_diffusion(F, kB):
    C = np.empty_like(F, dtype=np.uint8)

    C[-1] = (
        int(F[-1]) +
        int(kB[-1])
    ) & 255

    for i in range(len(F) - 2, -1, -1):
        C[i] = (
            int(F[i])
            + int(kB[i])
            + int(C[i + 1])
        ) & 255

    return C


CIPHER = backward_diffusion(F, kB)

cipher_image = CIPHER.reshape(
    H_img,
    W_img,
    C_img
) if C_img > 1 else CIPHER.reshape(
    H_img,
    W_img
)

print("Backward diffusion completed.")
print("Cipher image shape :", cipher_image.shape)

plt.figure(figsize=(7, 7))

if C_img == 1:
    plt.imshow(cipher_image, cmap="gray")
else:
    plt.imshow(
        cv2.cvtColor(
            cipher_image,
            cv2.COLOR_BGR2RGB
        )
    )

plt.title("Encrypted Image")
plt.axis("off")
plt.show()




def save_encrypted_image(cipher_image, original_path):
    base_name = os.path.splitext(
        os.path.basename(original_path)
    )[0]

    encrypted_path = f"{base_name}_encrypted.png"



    cv2.imwrite(
        encrypted_path,
        cipher_image
    )

    return encrypted_path


encrypted_path = save_encrypted_image(
    cipher_image,
    image_path
)

print("Encrypted image saved:", encrypted_path)

files.download(encrypted_path)

**DECRYPTION CODE**

In [ ]:
uploaded = files.upload()
encrypted_path = next(iter(uploaded))

encrypted_image = cv2.imread(
    encrypted_path,
    cv2.IMREAD_UNCHANGED
)

if encrypted_image is None:
    raise ValueError("Unable to read encrypted image.")

if encrypted_image.dtype != np.uint8:
    encrypted_image = np.clip(
        encrypted_image,
        0,
        255
    ).astype(np.uint8)

if encrypted_image.ndim == 2:
    H_dec, W_dec = encrypted_image.shape
    C_dec = 1
else:
    H_dec, W_dec, C_dec = encrypted_image.shape

N_dec = H_dec * W_dec * C_dec

C_dec_flat = encrypted_image.reshape(-1).copy()

print("================================================")
print("ENCRYPTED IMAGE")
print("================================================")
print("Shape    :", encrypted_image.shape)
print("Height   :", H_dec)
print("Width    :", W_dec)
print("Channels :", C_dec)
print("N        :", N_dec)
print("================================================")



KCNN_dec   = domain_kdf(Ks, "CNN")
KBNM_dec   = domain_kdf(Ks, "BNM")
KHENON_dec = domain_kdf(Ks, "HENON")
KP_dec     = domain_kdf(Ks, "PERM")
KF_dec     = domain_kdf(Ks, "FDIFF")
KB_dec     = domain_kdf(Ks, "BDIFF")



S_CNN_dec = generate_cnn_seed(KCNN_dec)




a_dec, b_dec = generate_dynamic_parameters(
    S_CNN_dec,
    N_dec
)



B_dec = generate_bnm_sequence(
    KBNM_dec,
    N_dec
)

KB_stream_dec = np.floor(
    B_dec * 256.0
).astype(np.uint8)



H_dec_seq = generate_henon_sequence(
    KHENON_dec,
    N_dec
)

KH_stream_dec = np.floor(
    (np.abs(H_dec_seq) % 1.0) * 256.0
).astype(np.uint8)



K_dec = np.bitwise_xor(
    KB_stream_dec,
    KH_stream_dec
)



kF_dec, kB_dec = regenerate_diffusion_streams(
    K_dec,
    N_dec
)



P_dec = fisher_yates_permutation(
    N_dec,
    KP_dec
)



def inverse_backward_diffusion(C, kB):

    F = np.empty_like(
        C,
        dtype=np.uint8
    )

    F[-1] = (
        int(C[-1])
        - int(kB[-1])
    ) & 255

    for i in range(
        len(C) - 2,
        -1,
        -1
    ):
        F[i] = (
            int(C[i])
            - int(kB[i])
            - int(C[i + 1])
        ) & 255

    return F


F_dec = inverse_backward_diffusion(
    C_dec_flat,
    kB_dec
)

print("1/5 Inverse backward diffusion completed.")



def inverse_forward_diffusion(F, kF):

    Z = np.empty_like(
        F,
        dtype=np.uint8
    )

    Z[0] = (
        int(F[0])
        - int(kF[0])
    ) & 255

    for i in range(
        1,
        len(F)
    ):
        Z[i] = (
            int(F[i])
            - int(kF[i])
            - int(F[i - 1])
        ) & 255

    return Z


Z_dec = inverse_forward_diffusion(
    F_dec,
    kF_dec
)

print("2/5 Inverse forward diffusion completed.")



def inverse_block_channel_permutation(
    Z,
    H,
    W,
    C,
    KP
):

    H = int(H)
    W = int(W)
    C = int(C)

    image = (
        Z.reshape(H, W, C)
        if C > 1
        else Z.reshape(H, W)
    )





    block_size = 16

    if H >= block_size and W >= block_size:

        h_blocks = H // block_size
        w_blocks = W // block_size
        total_blocks = h_blocks * w_blocks

        order = np.arange(
            total_blocks,
            dtype=np.int64
        )

        block_key = hashlib.sha256(
            KP + b"BLOCK"
        ).digest()

        for i in range(
            total_blocks - 1,
            0,
            -1
        ):

            j = int.from_bytes(
                hashlib.sha256(
                    block_key +
                    i.to_bytes(8, "big")
                ).digest()[:8],
                "big"
            ) % (i + 1)

            order[i], order[j] = (
                order[j],
                order[i]
            )

        result = image.copy()

        for dst, src in enumerate(order):

            dst_r = (
                dst // w_blocks
            ) * block_size

            dst_c = (
                dst % w_blocks
            ) * block_size

            src_r = (
                src // w_blocks
            ) * block_size

            src_c = (
                src % w_blocks
            ) * block_size

            result[
                src_r:src_r + block_size,
                src_c:src_c + block_size
            ] = image[
                dst_r:dst_r + block_size,
                dst_c:dst_c + block_size
            ]

        image = result





    if C > 1:

        channel_key = hashlib.sha256(
            KP + b"CHANNEL"
        ).digest()

        channel_order = np.arange(
            C,
            dtype=np.int64
        )

        for i in range(
            C - 1,
            0,
            -1
        ):

            j = int.from_bytes(
                channel_key[
                    i % 32:(i % 32) + 1
                ],
                "big"
            ) % (i + 1)

            channel_order[i], channel_order[j] = (
                channel_order[j],
                channel_order[i]
            )

        inverse_order = np.empty(
            C,
            dtype=np.int64
        )

        for i in range(C):
            inverse_order[
                channel_order[i]
            ] = i

        image = image[
            :,
            :,
            inverse_order
        ]

    return image.reshape(-1).astype(
        np.uint8
    )


Y_dec = inverse_block_channel_permutation(
    Z_dec,
    H_dec,
    W_dec,
    C_dec,
    KP_dec
)

print("3/5 Inverse block/channel permutation completed.")




def inverse_fisher_yates(Y, P):

    T = np.empty_like(
        Y,
        dtype=np.uint8
    )

    T[P] = Y

    return T


T_dec = inverse_fisher_yates(
    Y_dec,
    P_dec
)

print("4/5 Inverse Fisher-Yates completed.")




def inverse_modular_kernel_transform(
    T,
    a,
    b
):

    a_inv = np.empty(
        len(a),
        dtype=np.uint16
    )

    for i in range(len(a)):
        a_inv[i] = pow(
            int(a[i]),
            -1,
            256
        )

    X = (
        a_inv
        * (
            (
                T.astype(np.uint16)
                - b.astype(np.uint16)
            ) & 255
        )
    ) & 255

    return X.astype(np.uint8)


X_dec = inverse_modular_kernel_transform(
    T_dec,
    a_dec,
    b_dec
)

print("5/5 Inverse modular transformation completed.")


if C_dec == 1:

    decrypted_image = X_dec.reshape(
        H_dec,
        W_dec
    )

else:

    decrypted_image = X_dec.reshape(
        H_dec,
        W_dec,
        C_dec
    )




plt.figure(figsize=(7, 7))

if C_dec == 1:

    plt.imshow(
        decrypted_image,
        cmap="gray"
    )

else:

    plt.imshow(
        cv2.cvtColor(
            decrypted_image,
            cv2.COLOR_BGR2RGB
        )
    )

plt.title("Decrypted Image")
plt.axis("off")
plt.show()




if "image" in globals():

    original_for_check = image

    if original_for_check.shape == decrypted_image.shape:

        difference = np.abs(
            original_for_check.astype(
                np.int16
            )
            -
            decrypted_image.astype(
                np.int16
            )
        )

        max_difference = int(
            difference.max()
        )

        different_values = int(
            np.count_nonzero(
                difference
            )
        )

        exact_recovery = np.array_equal(
            original_for_check,
            decrypted_image
        )

        print()
        print("================================================")
        print("EXACT RECOVERY VERIFICATION")
        print("================================================")
        print(
            "Maximum difference :",
            max_difference
        )
        print(
            "Different values   :",
            different_values
        )
        print(
            "Exact recovery     :",
            exact_recovery
        )
        print("================================================")

    else:

        print("Original and decrypted image shapes differ.")



decrypted_path = "decrypted_original.png"

cv2.imwrite(
    decrypted_path,
    decrypted_image
)

print("Decrypted image saved:", decrypted_path)

files.download(
    decrypted_path
)